In [1]:
import torch
print(torch.cuda.get_device_name(0))


Tesla T4


In [1]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl huggingface_hub


In [2]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/finreport_dataset.jsonl",
    split="train"
)

print(dataset.column_names)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['instruction', 'input', 'output']


In [5]:
def format_example(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    }

dataset = dataset.map(format_example)
dataset = dataset.remove_columns(['instruction', 'input', 'output'])

print(dataset.column_names)


Map:   0%|          | 0/199 [00:00<?, ? examples/s]

['text']


In [6]:
import torch
torch.cuda.empty_cache()


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,   # VERY IMPORTANT
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully 🚀")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully 🚀


In [8]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,                    # Reduced from 16 → 8
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


In [9]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./mistral_finreport",
    per_device_train_batch_size=1,   # small batch
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=False,   # IMPORTANT
    bf16=False,   # IMPORTANT
)


In [10]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=sft_config,
)

print("Trainer initialized 🚀")


Adding EOS to train dataset:   0%|          | 0/199 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/199 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/199 [00:00<?, ? examples/s]

Trainer initialized 🚀


In [11]:
trainer.train()


Step,Training Loss
10,0.948555
20,0.287752
30,0.194421
40,0.184370
50,0.172064
60,0.154403
70,0.150555
80,0.157614
90,0.146668
100,0.148466


TrainOutput(global_step=125, training_loss=0.2331022834777832, metrics={'train_runtime': 2093.3564, 'train_samples_per_second': 0.475, 'train_steps_per_second': 0.06, 'total_flos': 1.118801013055488e+16, 'train_loss': 0.2331022834777832})

In [12]:
trainer.model.save_pretrained("/content/drive/MyDrive/mistral_finreport_adapter")
tokenizer.save_pretrained("/content/drive/MyDrive/mistral_finreport_adapter")

print("Adapter saved to Drive ✅")


Adapter saved to Drive ✅


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/mistral_finreport_adapter"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Inference model loaded 🚀")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Inference model loaded 🚀


In [5]:
def generate_financial_report(
    revenue,
    net_income,
    debt,
    growth,
    cashflow
):
    prompt = f"""### Instruction:
Generate a financial analysis report

### Input:
Revenue: {revenue}
Net Income: {net_income}
Debt: {debt}
YoY Growth: {growth}
Operating Cash Flow: {cashflow}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove the prompt and return only the generated report
    if "### Response:" in decoded:
        return decoded.split("### Response:")[-1].strip()
    else:
        return decoded.strip()


In [6]:
report = generate_financial_report(
    revenue="$50B",
    net_income="$6B",
    debt="$20B",
    growth="8%",
    cashflow="Positive"
)

print(report)


Financial Analysis Report

1. Revenue & Growth Analysis:
The company reported revenue of $50B with a year-over-year growth of 8%, indicating moderate growth.

2. Profitability Assessment:
Net income stands at $6B, reflecting solid profitability.

3. Leverage & Risk Analysis:
Total debt amounts to $20B, suggesting elevated financial leverage.

4. Liquidity Position:
Operating cash flow is $7.65B, indicating healthy liquidity.

5. Overall Risk Score: 40/100

6. Investment Recommendation: Buy

7. Final Outlook:
Based on current financial metrics, the company presents a buy investment profile with balanced growth and risk factors.


In [7]:
import gradio as gr

def generate_report_ui(revenue, net_income, debt, growth, cashflow):
    return generate_financial_report(
        revenue,
        net_income,
        debt,
        growth,
        cashflow
    )

demo = gr.Interface(
    fn=generate_report_ui,
    inputs=[
        gr.Textbox(label="Revenue (e.g., $50B)"),
        gr.Textbox(label="Net Income (e.g., $6B)"),
        gr.Textbox(label="Debt (e.g., $20B)"),
        gr.Textbox(label="YoY Growth (e.g., 8%)"),
        gr.Textbox(label="Operating Cash Flow (Positive/Negative)")
    ],
    outputs=gr.Textbox(label="Generated Financial Report"),
    title="AI Financial Analysis Generator",
    description="Fine-tuned Mistral-7B using QLoRA (4-bit) for structured financial report generation."
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff98db0ff27416ab59.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
!pip install -q huggingface_hub


In [13]:
from huggingface_hub import logout
logout()


In [14]:
from huggingface_hub import login
login()


In [15]:
repo_id = "thanmayee22/mistral-financial-qlora"

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print("Adapter pushed successfully 🚀")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   9%|8         | 1.21MB / 13.6MB            

README.md: 0.00B [00:00, ?B/s]

Adapter pushed successfully 🚀
